# 08 · Explain a shift with genes, programs and response curves

**ScGeo 0.2.0 learning track.** Use continuous simulated expression values and a fitted, explicitly supplied affine PCA transformation. The target is descriptive attribution to a measured representation displacement, not causal driver discovery.

Use the [separate tutorial environment](README.md). Run cells in order in a fresh kernel. Outputs are written to a new ignored results directory; manuscript inputs and outputs are not used.

In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import scgeo as sg
from IPython.display import display

cwd = Path.cwd().resolve()
root = next((p for p in (cwd, *cwd.parents)
             if (p / "scripts/scgeo_v020_examples/SOURCE_PART1.json").is_file()), None)
if root is None:
    raise RuntimeError("Open this notebook from the scgeo-notebooks checkout.")
examples = root / "scripts/scgeo_v020_examples"
if str(examples) not in sys.path:
    sys.path.insert(0, str(examples))  # only tutorial helpers, not package source
from tutorial_support import setup, save_preview, show_gallery, FORMATS
out = setup(root, "08_shift_explanations")

In [ ]:
from explain_shift_gallery import make_example_data, run_gallery, composition_only_example, distributional_counterexample
adata, programs, cell_truth, feature_truth = make_example_data(seed=37)
display(feature_truth)
sg.tl.robust_shift(adata, rep="X_pca", condition_key="condition", group0="control",
    group1="treated", by="state", sample_key="sample", center="geometric_median",
    n_boot=0, seed=37)
explanation = sg.tl.explain_shift(adata, shift_key="robust_shift", loadings_key="PCs",
    modules=programs, local_key="region", min_cells=10, min_samples=2,
    n_boot=80, seed=37)
display(explanation["summary"])
display(explanation["local_genes"].head())

## Gene and program curves

The horizontal axis is a geometric shift coordinate. It is not pseudotime. Per-sample traces expose support and consistency. Contribution maps use the verified feature-to-coordinate mapping; genes not in that map cannot be attributed by this calculation.

In [ ]:
result = sg.pl.shift_curves(adata, var_names=["Response_1","LocalResponse_1","Identity_1"],
    state="Simulated state", basis="display", panels=("curve","expression","contribution"),
    return_result=True, show=False)
save_preview(result, out, "gene_curves")
result = sg.pl.shift_curves(adata, program_names=["Response","LocalResponse"],
    state="Simulated state", basis="display", panels=("curve",),
    return_result=True, show=False)
save_preview(result, out, "program_curves")
result = sg.pl.shift_embedding(adata, states=["Simulated state"], basis="display",
    color="condition", return_result=True, show=False)
save_preview(result, out, "curved_state_shift")

## Test the interpretation, not just the plot

A composition-only shift and a joint-distribution change with unchanged marginal means are different estimands. The release gallery additionally checks signed attribution, rotation/duplication invariance, local opposing responses and zero shifts. Supplied regulatory evidence prioritizes candidates, not causal claims.

In [ ]:
benchmark = run_gallery(out / "gallery", formats=FORMATS, seed=37, n_boot=80)
display(benchmark["checks"])
display(benchmark["metrics"])
show_gallery(out / "gallery", limit=2)

See [explain shift](https://github.com/liuifrec/scgeo/blob/v0.2.0/docs/EXPLAIN_SHIFT.md) for affine-map requirements, local support, gene/program overlap and interpretation limits.